# Demo notebook
Event-Market Graph: what the graph holds and how events and prices connect through `TradingDay`.

**Before running:** `docker compose up -d neo4j`, then `make schema` and `make ingest`. Educational use only: a coincidence in time is never a proof of cause.

In [ ]:
# Connect to Neo4j with the project's settings
from app.database.connection import get_driver, check_connection
import pandas as pd
import matplotlib.pyplot as plt

def query(cypher, **params):
    """Run a read query and return a DataFrame."""
    with get_driver().session() as session:
        return pd.DataFrame([r.data() for r in session.run(cypher, **params)])

print('Neo4j reachable:', check_connection())

## 1. Graph overview
Nodes per label and relationships per type (same numbers as `make stats`).

In [ ]:
nodes = query('MATCH (n) UNWIND labels(n) AS label RETURN label, count(*) AS n ORDER BY n DESC')
rels = query('MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n ORDER BY n DESC')
display(nodes)
display(rels)

## 2. Events by theme and by country
Which kinds of events and which countries dominate the selected window.

In [ ]:
themes = query('MATCH (e:Event)-[:HAS_THEME]->(t:Theme) RETURN t.label AS theme, count(e) AS events ORDER BY events DESC')
countries = query('MATCH (e:Event)-[:OCCURRED_IN]->(c:Country) RETURN c.name AS country, count(e) AS events ORDER BY events DESC LIMIT 10')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
themes.plot.barh(x='theme', y='events', ax=axes[0], legend=False, title='Events by theme')
countries.plot.barh(x='country', y='events', ax=axes[1], legend=False, title='Top countries')
plt.tight_layout()
plt.show()

## 3. A price series and the central-bank decisions
Brent crude oil with a vertical line on every monetary-policy event (derived from the Fed and ECB policy rates).

In [ ]:
brent = query("MATCH (:PriceSeries {series_id: 'BRENT'})-[p:PRICE_ON]->(d:TradingDay) RETURN d.date.year AS y, d.date.month AS m, d.date.day AS d, p.close AS close ORDER BY y, m, d")
brent['date'] = pd.to_datetime(dict(year=brent.y, month=brent.m, day=brent.d))
cb = query("MATCH (e:Event)-[:HAS_THEME]->(:Theme {code: 'MONETARY_POLICY'}) RETURN e.date.year AS y, e.date.month AS m, e.date.day AS d, e.title AS title ORDER BY y, m, d")
cb['date'] = pd.to_datetime(dict(year=cb.y, month=cb.m, day=cb.d))
ax = brent.plot(x='date', y='close', figsize=(11, 4), legend=False, title='Brent (USD per barrel) and central-bank rate changes')
for day in cb['date']:
    ax.axvline(day, color='red', alpha=0.4)
plt.show()
display(cb[['date', 'title']])

## 4. Joining events and prices through TradingDay
The five largest daily moves of Brent, and how many tension events happened in the three days up to each. This is descriptive: it counts coincidences and says nothing about causes.

In [ ]:
moves = query('''
MATCH (:PriceSeries {series_id: 'BRENT'})-[p:PRICE_ON]->(d:TradingDay)
WHERE p.ret_1d IS NOT NULL
WITH d, p ORDER BY abs(p.ret_1d) DESC LIMIT 5
OPTIONAL MATCH (e:Event)-[:ON_DAY]->(d2:TradingDay)
WHERE d2.date <= d.date AND d2.date >= d.date - duration({days: 3}) AND NOT (e)-[:HAS_THEME]->(:Theme {code: 'MONETARY_POLICY'})
RETURN toString(d.date) AS day, round(p.ret_1d * 10000) / 100 AS move_pct, count(e) AS tension_events
ORDER BY abs(move_pct) DESC
''')
display(moves)

## 5. Where the other stages plug in
Stage 2 adds graph algorithms (PageRank, Louvain) and benchmarks on this graph, stage 3 exposes it through the API and the agent. This notebook only explores the data model.